In [13]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio import windows
import geopandas as gpd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import SegformerForSemanticSegmentation
from tqdm import tqdm
import segmentation_models_pytorch as smp


In [14]:
# ============================ НАСТРОЙКИ ============================
PICS_DIR = r'D:\kanopus_ikutsk\захламление_new'
MASK_DIR = r'D:\kanopus_ikutsk\захламление_main_dataset'

PATCH_SIZE = 512
STRIDE = 256
BATCH_SIZE = 8
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
HARD_NEG_WEIGHT = 2.0   # вес hard-negative патчей в сэмплере
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {DEVICE}')

Используется устройство: cuda


In [15]:


# ============================ ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ============================
TARGET_PREFIXES = ['hlam3_', 'hlam2_', 'Olskij_', 'irkutsk_']
HN_PREFIX = 'hn_'

def get_file_pairs(pics_dir, mask_dir):
    """
    Возвращает список троек (tif_path, target_geojson_or_None, hn_geojson_or_None).
    Целевая маска выбирается по одному из префиксов TARGET_PREFIXES.
    Hard-negative маска — по префиксу 'hn_'.
    """
    pairs = []
    for fname in os.listdir(pics_dir):
        if fname.lower().endswith(('.tif', '.tiff')):
            base = os.path.splitext(fname)[0]
            target_path = None
            for prefix in TARGET_PREFIXES:
                p = os.path.join(mask_dir, prefix + base + '.geojson')
                if os.path.exists(p):
                    target_path = p
                    break
            hn_path = os.path.join(mask_dir, HN_PREFIX + base + '.geojson')
            if not os.path.exists(hn_path):
                hn_path = None
            if target_path is not None or hn_path is not None:
                pairs.append((os.path.join(pics_dir, fname), target_path, hn_path))
    return pairs

def normalize_image(img):
    img = img.astype(np.float32)
    for c in range(img.shape[0]):
        min_val = img[c].min()
        max_val = img[c].max()
        if max_val - min_val > 1e-6:
            img[c] = (img[c] - min_val) / (max_val - min_val)
        else:
            img[c] = 0
    return img

def geojson_to_mask_for_window(geojson_path, transform, out_shape):
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        return np.zeros(out_shape, dtype=np.uint8)
    shapes = [(geom, 1) for geom in gdf.geometry]
    mask = rasterize(shapes, out_shape=out_shape, transform=transform,
                     fill=0, dtype='uint8')
    return mask

# ============================ ГЕНЕРАЦИЯ КООРДИНАТ ============================
def build_all_patch_coords(pairs, patch_size, stride, context=0):
    coords = []
    for tif_path, target_path, hn_path in pairs:
        with rasterio.open(tif_path) as src:
            width, height = src.width, src.height
        for y in range(context, height - patch_size - context + 1, stride):
            for x in range(context, width - patch_size - context + 1, stride):
                coords.append((tif_path, target_path, hn_path, x, y))
    return coords

# ============================ ДАТАСЕТ ============================
class OnTheFlyDataset(Dataset):
    def __init__(self, patch_coords, patch_size, transform=None,
                 return_positive_info=False, return_hn_info=False):
        self.patch_coords = patch_coords
        self.patch_size = patch_size
        self.transform = transform
        self.return_positive_info = return_positive_info
        self.return_hn_info = return_hn_info

    def __len__(self):
        return len(self.patch_coords)

    def __getitem__(self, idx):
        tif_path, target_path, hn_path, x, y = self.patch_coords[idx]

        with rasterio.open(tif_path) as src:
            window = windows.Window(x, y, self.patch_size, self.patch_size)
            img = src.read(window=window)
            win_transform = src.window_transform(window)

        # Целевая маска
        if target_path is not None:
            mask = geojson_to_mask_for_window(target_path, win_transform,
                                              (self.patch_size, self.patch_size))
        else:
            mask = np.zeros((self.patch_size, self.patch_size), dtype=np.uint8)

        # Hard-negative маска: переопределяет класс 1 -> класс 0
        if hn_path is not None:
            hn_mask = geojson_to_mask_for_window(hn_path, win_transform,
                                                 (self.patch_size, self.patch_size))
            mask[hn_mask == 1] = 0
            hn_ratio = float(hn_mask.mean())
        else:
            hn_ratio = 0.0

        img = normalize_image(img)
        img = torch.tensor(img, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        if self.transform:
            img, mask = self.transform(img, mask)

        pos_ratio = (mask == 1).float().mean().item()

        if self.return_positive_info and self.return_hn_info:
            return img, mask, pos_ratio, hn_ratio
        elif self.return_positive_info:
            return img, mask, pos_ratio
        elif self.return_hn_info:
            return img, mask, hn_ratio
        return img, mask

# ============================ ВЕСА И СЭМПЛЕР ============================
def compute_class_weights_and_sampler(dataset, hard_neg_indices=None,
                                      hard_neg_weight=HARD_NEG_WEIGHT):
    positive_ratios = []
    labels_present = []
    print("Сканирование train-патчей для вычисления статистики...")
    for i in tqdm(range(len(dataset))):
        _, mask, pos_ratio = dataset[i]
        positive_ratios.append(pos_ratio)
        labels_present.append(1 if pos_ratio > 0.001 else 0)

    positive_ratios = np.array(positive_ratios)
    labels_present = np.array(labels_present)

    sum_pos = positive_ratios.sum()
    sum_neg = (1 - positive_ratios).sum()
    weight_pos = 1.0 / (sum_pos + 1e-6)
    weight_neg = 1.0 / (sum_neg + 1e-6)
    mean_weight = (weight_pos + weight_neg) / 2
    weight_pos /= mean_weight
    weight_neg /= mean_weight
    class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float32)

    # Базовые веса сэмплера
    sample_weights = np.where(labels_present == 1, 7.0, 1.0).astype(np.float64)
    # Усиление hard-negative патчей
    if hard_neg_indices is not None:
        for idx in hard_neg_indices:
            # не ослабляем положительные патчи
            sample_weights[idx] = max(sample_weights[idx], hard_neg_weight)
    sample_weights = sample_weights / sample_weights.sum()
    sampler = WeightedRandomSampler(sample_weights,
                                    num_samples=len(dataset),
                                    replacement=True)
    return class_weights, sampler

def find_hard_negative_indices(dataset):
    """Возвращает список индексов патчей, содержащих hard-negative пиксели."""
    hn_indices = []
    print("Поиск hard-negative патчей...")
    for i in tqdm(range(len(dataset))):
        _, _, hn_ratio = dataset[i]
        if hn_ratio > 0.0:
            hn_indices.append(i)
    return hn_indices


In [16]:
# ============================ ПОДГОТОВКА ДАННЫХ ============================
pairs = get_file_pairs(PICS_DIR, MASK_DIR)
print(f'Найдено пар файлов: {len(pairs)}')
n_target = sum(1 for _, t, _ in pairs if t is not None)
n_hn = sum(1 for _, _, h in pairs if h is not None)
print(f'  с целевой маской: {n_target}, с hard-negative: {n_hn}')

Найдено пар файлов: 42
  с целевой маской: 40, с hard-negative: 2


In [17]:

# ============================ ПОДГОТОВКА ДАННЫХ ============================
pairs = get_file_pairs(PICS_DIR, MASK_DIR)
print(f'Найдено пар файлов: {len(pairs)}')
n_target = sum(1 for _, t, _ in pairs if t is not None)
n_hn = sum(1 for _, _, h in pairs if h is not None)
print(f'  с целевой маской: {n_target}, с hard-negative: {n_hn}')

all_coords = build_all_patch_coords(pairs, PATCH_SIZE, STRIDE)
print(f'Всего возможных патчей: {len(all_coords)}')

train_coords, temp_coords = train_test_split(all_coords, test_size=0.4, random_state=42)
val_coords, test_coords = train_test_split(temp_coords, test_size=0.5, random_state=42)
print(f'Train патчей: {len(train_coords)}, Val: {len(val_coords)}, Test: {len(test_coords)}')

# Сканирование train
train_dataset_info = OnTheFlyDataset(train_coords, PATCH_SIZE, return_positive_info=True)
train_dataset_hn = OnTheFlyDataset(train_coords, PATCH_SIZE, return_hn_info=True)
hard_neg_indices = find_hard_negative_indices(train_dataset_hn)
print(f'Hard-negative патчей в train: {len(hard_neg_indices)} из {len(train_coords)}')

Найдено пар файлов: 42
  с целевой маской: 40, с hard-negative: 2
Всего возможных патчей: 16050
Train патчей: 9630, Val: 3210, Test: 3210
Поиск hard-negative патчей...


100%|██████████| 9630/9630 [02:59<00:00, 53.50it/s]

Hard-negative патчей в train: 126 из 9630


In [18]:


class_weights, train_sampler = compute_class_weights_and_sampler(
    train_dataset_info,
    hard_neg_indices=hard_neg_indices,
    hard_neg_weight=HARD_NEG_WEIGHT
)

train_dataset = OnTheFlyDataset(train_coords, PATCH_SIZE)
val_dataset = OnTheFlyDataset(val_coords, PATCH_SIZE)
test_dataset = OnTheFlyDataset(test_coords, PATCH_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ============================ МОДЕЛЬ ============================
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=2,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)
first_conv = None
for module in model.modules():
    if isinstance(module, nn.Conv2d) and module.in_channels == 3:
        first_conv = module
        break
if first_conv is None:
    raise ValueError("Не найден свёрточный слой с 3 входными каналами")
new_conv = nn.Conv2d(4, first_conv.out_channels,
                     kernel_size=first_conv.kernel_size,
                     stride=first_conv.stride, padding=first_conv.padding,
                     bias=first_conv.bias is not None)
with torch.no_grad():
    new_conv.weight[:, :3] = first_conv.weight
    new_conv.weight[:, 3] = first_conv.weight[:, 0]
for name, module in model.named_modules():
    if module is first_conv:
        parent_name = name.rsplit('.', 1)[0] if '.' in name else ''
        parent = model.get_submodule(parent_name) if parent_name else model
        attr_name = name.rsplit('.', 1)[-1] if '.' in name else name
        setattr(parent, attr_name, new_conv)
        print(f"Заменён слой: {name}")
        break
model.config.num_channels = 4
model.to(DEVICE)
print(f'Модель загружена: {model.config.num_labels} классов, {model.config.num_channels} каналов')

Сканирование train-патчей для вычисления статистики...


100%|██████████| 9630/9630 [02:55<00:00, 54.74it/s]
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `150`.
Loading weights: 100%|██████████| 208/208 [00:00<00:00, 12235.32it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([2, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([2])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weig

Заменён слой: segformer.stages.0.patch_embeddings.proj
Модель загружена: 2 классов, 4 каналов


In [19]:


# ============================ LOSS, OPTIMIZER ============================
tversky_loss = smp.losses.TverskyLoss(mode='multiclass', alpha=0.7, beta=0.4, from_logits=True)
ce_loss = torch.nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

def combined_loss(logits, targets):
    return 0.15 * ce_loss(logits, targets) + 0.85 * tversky_loss(logits, targets)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

In [20]:


# ============================ ОБУЧЕНИЕ ============================
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]')
    for images, masks in pbar:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        logits = outputs.logits
        logits = torch.nn.functional.interpolate(logits, size=masks.shape[-2:],
                                                mode='bilinear', align_corners=False)
        loss = combined_loss(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        pbar.set_postfix({'loss': loss.item()})
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            logits = outputs.logits
            logits = torch.nn.functional.interpolate(logits, size=masks.shape[-2:],
                                                    mode='bilinear', align_corners=False)
            loss = combined_loss(logits, masks)
            val_loss += loss.item() * images.size(0)
    val_loss /= len(val_loader.dataset)
    print(f'Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}')
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_segformer_hlam_hn.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print('Ранняя остановка.')
            break


Epoch 1/20 [train]: 100%|██████████| 1204/1204 [18:07<00:00,  1.11it/s, loss=0.245]   


Epoch 1: Train Loss = 0.2619, Val Loss = 0.0611


Epoch 2/20 [train]: 100%|██████████| 1204/1204 [18:11<00:00,  1.10it/s, loss=0.35]    


Epoch 2: Train Loss = 0.1360, Val Loss = 0.0622


Epoch 3/20 [train]: 100%|██████████| 1204/1204 [18:18<00:00,  1.10it/s, loss=0.000108]


Epoch 3: Train Loss = 0.1134, Val Loss = 0.0407


Epoch 4/20 [train]:   3%|▎         | 32/1204 [00:30<18:26,  1.06it/s, loss=0.128]   


KeyboardInterrupt: 

In [21]:

# ============================ ОЦЕНКА ============================
model.load_state_dict(torch.load('best_model_segformer_hlam_hn.pth'))
model.eval()

all_preds = []
all_targets = []
with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Оценка на тесте'):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        outputs = model(images)
        logits = outputs.logits
        logits = torch.nn.functional.interpolate(logits, size=masks.shape[-2:],
                                                mode='bilinear', align_corners=False)
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.cpu().numpy().flatten())
        all_targets.append(masks.cpu().numpy().flatten())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

print('\nОтчёт по классам на тестовом наборе:')
print(classification_report(all_targets, all_preds, target_names=['фон', 'захламление']))

C:\Users\user\AppData\Local\Temp\ipykernel_2260\2337989392.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_segformer_hlam_hn


Отчёт по классам на тестовом наборе:
              precision    recall  f1-score   support

         фон       1.00      1.00      1.00 838589904
 захламление       0.58      0.95      0.72   2892336

    accuracy                           1.00 841482240
   macro avg       0.79      0.98      0.86 841482240
weighted avg       1.00      1.00      1.00 841482240

